# DrugCLIP screening from an AlphaFold structure

For a target with **no experimental structure** - here human PGK2 (P07205),
which has none in the PDB.

The problem: a pocket is defined as the residues within 6 A of a bound ligand,
and a predicted model has no ligand. The screen-pipeline README is explicit
about the preferred fix:

> We recommend you use experimentally solved ligands to define the pocket, even
> if you would like to try DrugCLIP with AlphaFold2 models (**you can align
> models first**). Tools like Fpocket can be used, but only half of those
> pockets are precise enough for DrugCLIP.

So instead of *detecting* a pocket, we **borrow one** from a solved relative:

```
AlphaFold human PGK2        2PAA (mouse Pgk2, ATP bound)
        |                            |
        +--------- align ------------+
                     |
          apply the same transform to ATP alone
                     |
   human model + ATP in its binding site = pseudo-holo
                     |
            pocket_from_pdb.py, unchanged
```

Nothing downstream knows the ligand was copied in. **Stages 5-7 are identical to
`run_screen.ipynb`.**

On the paper's benchmarks this route (`Exp. Pocket`) reaches EF1% ~24 on
AlphaFold2 structures, against 19.0 for Fpocket.

## Configuration

In [ ]:
from pathlib import Path

REPO     = Path("/root/GitRepos/DrugCLIP")
PIPELINE = Path("/root/GitRepos/DrugCLIP_screen_pipeline")
WORK     = REPO / "data" / "pgk2_af"          # gitignored

UNIPROT   = "P07205"      # human PGK2 - no experimental structure exists
HOLO_PDB  = "2PAA"        # mouse Pgk2, holo. Ligand donor.
HOLO_CHAIN = "A"
HET_ID    = "ATP"         # which ligand defines the pocket

MOL_CSV  = REPO / "data" / "pgk2" / "sample_molecules.csv"
N_CONF   = 10
WEIGHTS  = REPO / "model" / "dude_identity_90.pt"
TMALIGN  = REPO / "HomoAug" / "bin" / "TMalign"   # shipped with the repo

POCKET_LMDB = WORK / "pocket.lmdb"
MOLS_LMDB   = WORK / "mols.lmdb"
EMB_DIR     = WORK / "emb"

WORK.mkdir(parents=True, exist_ok=True)
EMB_DIR.mkdir(parents=True, exist_ok=True)
print("work dir:", WORK)
print("weights: ", "found" if WEIGHTS.exists() else "MISSING")

In [ ]:
import sys, json, pickle, subprocess, shutil, urllib.request
import numpy as np, pandas as pd, lmdb, torch
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog("rdApp.*")

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "usage"))
from transplant import transplant          # helper that lives beside this notebook

print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())

## Stage 1 - AlphaFold model

**You do not need to install or run AlphaFold.** The AlphaFold Database holds
pre-computed structures for essentially every human protein; this is a download.

The file version changes over time, so the URL is resolved from the API rather
than hardcoded.

plDDT (AlphaFold's per-residue confidence, 0-100) is stored in the B-factor
column. Above 90 is very high, below 70 is unreliable.

In [ ]:
api = "https://alphafold.ebi.ac.uk/api/prediction/" + UNIPROT
meta = json.loads(urllib.request.urlopen(api).read())[0]
print("gene:", meta.get("gene"), "| length:", meta.get("uniprotEnd"),
      "| model version:", meta.get("latestVersion"))

af_pdb = WORK / "AF_model.pdb"
if not af_pdb.exists():
    urllib.request.urlretrieve(meta["pdbUrl"], af_pdb)
    print("downloaded", meta["pdbUrl"].rsplit("/", 1)[-1])

b = np.array([float(l[60:66]) for l in open(af_pdb) if l.startswith("ATOM")])
print()
print("plDDT  median %.1f  mean %.1f" % (np.median(b), b.mean()))
print("  very high (>90): %.1f%%" % (100 * (b > 90).mean()))
print("  low       (<70): %.1f%%" % (100 * (b < 70).mean()))
if (b < 70).mean() > 0.3:
    print("WARNING: large low-confidence fraction - treat the pocket with caution")

## Stage 2 - Holo reference (the ligand donor)

A solved structure of a close relative, with a ligand in the site you care about.

Human PGK2 has no PDB entry, so we use **mouse Pgk2 (2PAA)**, which was solved
with both ATP and 3PG bound. Human PGK1 structures (e.g. 2XE6) are an
alternative donor - same species, different isoform.

In [ ]:
holo = WORK / (HOLO_PDB + ".pdb")
if not holo.exists():
    urllib.request.urlretrieve(
        "https://files.rcsb.org/download/" + HOLO_PDB + ".pdb", holo)
    print("downloaded", HOLO_PDB)

het = {}
for l in open(holo):
    if l.startswith("HETATM"):
        k = l[17:20].strip()
        het[k] = het.get(k, 0) + 1
print("ligands in", HOLO_PDB + ":")
for k, v in sorted(het.items(), key=lambda x: -x[1]):
    print("   %-5s %3d atoms%s" % (k, v, "   <-- donor" if k == HET_ID else ""))
assert HET_ID in het, HET_ID + " not present"

## Stage 3 - Align and transplant

`TMalign` superposes the holo structure onto the model and reports the transform
as a rotation `u` and translation `t`. Applying that same transform to the
ligand alone moves it into the model's coordinate frame.

The binary ships with this repo (`HomoAug/bin/TMalign`), and `HomoAug` already
uses exactly this operation for data augmentation.

**This step can fail silently.** Apply the transform backwards and you still get
a valid PDB - the ligand just lands somewhere wrong, and the pocket extracted
around it will be meaningless rather than empty. Hence the checks below.

In [ ]:
TMALIGN.chmod(0o755)                     # the shipped binary may lack +x

info = transplant(model_pdb=af_pdb, holo_pdb=holo, het_id=HET_ID,
                  tmalign=TMALIGN, out_pdb=WORK / "AF_model_with_lig.pdb",
                  holo_chain=HOLO_CHAIN, workdir=WORK)

print("TM-score            %.4f" % info["tm_score"])
print("ligand atoms        %d"   % info["ligand_atoms"])
print("min dist to protein %.2f A  (before transform: %.2f A)"
      % (info["min_dist_after"], info["min_dist_before"]))
print("protein atoms <6 A  %d"   % info["contacts_after"])

In [ ]:
# --- verification -------------------------------------------------------
ok = True

if info["tm_score"] < 0.5:
    print("FAIL  TM-score %.2f - the two structures are not the same fold"
          % info["tm_score"]); ok = False
else:
    print("pass  TM-score %.2f (>0.5 means same fold; >0.9 is near-identical)"
          % info["tm_score"])

d = info["min_dist_after"]
if d > 5:
    print("FAIL  nearest protein atom is %.1f A away - the ligand is floating "
          "outside the protein. The transform is probably inverted." % d); ok = False
elif d < 1.0:
    print("FAIL  nearest protein atom is %.1f A - severe clash" % d); ok = False
else:
    print("pass  closest contact %.2f A (a real contact is ~2-4 A)" % d)

if info["contacts_after"] < 30:
    print("FAIL  only %d protein atoms within 6 A - not a buried site"
          % info["contacts_after"]); ok = False
else:
    print("pass  %d protein atoms within 6 A" % info["contacts_after"])

print()
print("TRANSPLANT OK" if ok else "TRANSPLANT FAILED - do not trust what follows")

## Stage 4 - Extract the pocket

Identical to the experimental-structure notebook: the file now looks like any
other holo complex.

Compare the atom count against the holo reference. A pocket of a very different
size suggests the model's conformation differs from the solved one - for PGK
specifically, that would most likely be the two-domain hinge sitting more open.

In [ ]:
pdb_dir = WORK / "pdb_in"
if pdb_dir.exists():
    shutil.rmtree(pdb_dir)
pdb_dir.mkdir()
shutil.copy(WORK / "AF_model_with_lig.pdb", pdb_dir / ("AFmodel_" + HET_ID + ".pdb"))

r = subprocess.run([sys.executable, str(PIPELINE / "pocket_from_pdb.py"),
                    str(pdb_dir), "--name", "pocket"],
                   capture_output=True, text=True, cwd=str(PIPELINE))
print(r.stdout or "(no stdout)")
if r.returncode != 0:
    print("STDERR:", r.stderr[-1500:])

produced = pdb_dir / "pocket.lmdb"
assert produced.exists(), "pocket.lmdb not created"
if POCKET_LMDB.exists():
    POCKET_LMDB.unlink()
shutil.move(str(produced), str(POCKET_LMDB))


def read_lmdb(path):
    env = lmdb.open(str(path), subdir=False, readonly=True, lock=False,
                    readahead=False, meminit=False, max_readers=256)
    with env.begin() as txn:
        out = [pickle.loads(v) for _, v in txn.cursor()]
    env.close()
    return out

pockets = read_lmdb(POCKET_LMDB)
assert pockets, ("pocket.lmdb is EMPTY - pocket_from_pdb.py swallows every "
                 "error in a bare except, so this is what a parse failure looks like")
for p in pockets:
    print("%-26s %4d atoms  elements=%s"
          % (p["pocket"], len(p["pocket_atoms"]), sorted(set(p["pocket_atoms"]))))

In [ ]:
# compare against the same pocket taken from the experimental structure
ref = REPO / "data" / "pgk2" / "pocket.lmdb"
n_af = len(pockets[0]["pocket_atoms"])
if ref.exists():
    n_ref = len(read_lmdb(ref)[0]["pocket_atoms"])
    print("AlphaFold + transplant : %d atoms" % n_af)
    print("experimental (%s)     : %d atoms" % (HOLO_PDB, n_ref))
    print("difference             : %+.0f%%" % (100 * (n_af - n_ref) / n_ref))
    if abs(n_af - n_ref) / n_ref > 0.35:
        print("\nWARNING: sizes differ by more than a third. For PGK this would "
              "most likely mean the hinge is in a different state.")
    else:
        print("\nComparable - the two pockets describe the same site.")
else:
    print("AlphaFold pocket: %d atoms (no experimental pocket to compare)" % n_af)

## Stage 5 - Molecule library

Unchanged from `run_screen.ipynb`. SMILES -> add hydrogens -> embed conformers
-> MMFF optimise -> drop hydrogens.

In [ ]:
df = pd.read_csv(MOL_CSV)
print(len(df), "molecules from", MOL_CSV.name)

def to_record(name, smi, n_conf=N_CONF):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, "unparseable SMILES"
    mol = Chem.AddHs(mol)
    cids = AllChem.EmbedMultipleConfs(mol, numConfs=n_conf, randomSeed=42,
                                      pruneRmsThresh=1)
    if len(cids) == 0:
        return None, "no conformer embedded"
    try:
        AllChem.MMFFOptimizeMoleculeConfs(mol)
    except Exception:
        pass
    mol = Chem.RemoveHs(mol)
    return {"atoms": [a.GetSymbol() for a in mol.GetAtoms()],
            "coordinates": [np.array(mol.GetConformer(c).GetPositions(),
                                     dtype=np.float32)
                            for c in range(mol.GetNumConformers())],
            "smi": smi, "name": name}, None

records, failures = [], []
for _, row in df.iterrows():
    rec, err = to_record(row["name"], row["smiles"])
    (records.append(rec) if rec else failures.append((row["name"], err)))
print("built", len(records), "| failed", len(failures))
for f in failures:
    print("  FAILED:", f)

vocab = set((REPO / "data" / "dict_mol.txt").read_text().split())
unknown = set(a for r in records for a in r["atoms"]) - vocab
print("elements not in dict_mol.txt (would become [UNK]):", sorted(unknown) or "none")

if MOLS_LMDB.exists():
    MOLS_LMDB.unlink()
env = lmdb.open(str(MOLS_LMDB), subdir=False, readonly=False, lock=False,
                readahead=False, meminit=False, map_size=1099511627776)
with env.begin(write=True) as txn:
    for i, r in enumerate(records):
        txn.put(str(i).encode("ascii"), pickle.dumps(r))
env.close()
print(MOLS_LMDB.name, ":", len(read_lmdb(MOLS_LMDB)), "records")

## Stage 6 - Screen

The embedding cache is keyed on **filename, not contents**. Changing the library
without clearing it would silently score the previous molecules, so it is cleared
here.

In [ ]:
for f in EMB_DIR.glob("*.pkl"):
    f.unlink()
    print("cleared stale cache:", f.name)

assert WEIGHTS.exists(), "weights missing: %s" % WEIGHTS
cmd = [sys.executable, "./unimol/retrieval.py",
       "--user-dir", "./unimol", str(REPO / "data"),
       "--valid-subset", "test",
       "--results-path", str(WORK / "results"),
       "--num-workers", "0", "--ddp-backend", "c10d", "--batch-size", "8",
       "--task", "drugclip", "--loss", "in_batch_softmax", "--arch", "drugclip",
       "--max-pocket-atoms", "256", "--seed", "1",
       "--path", str(WEIGHTS),
       "--log-interval", "100", "--log-format", "simple",
       "--mol-path", str(MOLS_LMDB),
       "--pocket-path", str(POCKET_LMDB),
       "--emb-dir", str(EMB_DIR)]
if not torch.cuda.is_available():
    cmd.append("--cpu")

r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO))
print(r.stdout[-1500:])
if r.returncode != 0:
    print("--- STDERR ---")
    print(r.stderr[-3000:])

## Stage 7 - Results

These are cosine similarities. The **ordering** is meaningful; the absolute
values are not, without a background distribution. For a real campaign, spike
the compounds of interest into 10k-100k background molecules and use the
z-score (> 3 per the screen-pipeline README, > 4 in the Science paper).

In [ ]:
out = EMB_DIR / "ranked_compounds.txt"
if out.exists():
    smi2name = dict((r["smi"], r["name"]) for r in records)
    rows = []
    for line in out.read_text().splitlines():
        if line.strip():
            smi, score = line.rsplit("\t", 1)
            rows.append({"name": smi2name.get(smi, "?"),
                         "score": float(score), "smiles": smi})
    af = pd.DataFrame(rows)
    display(af)
else:
    print("not produced:", out)